In [1]:
import pandas as pd
import numpy as np
import sqlite3

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [2]:
# Indlæs datasæt med manglende værdier
df = pd.read_csv("Sales_with_NaNs_v1.3.csv")

df.head()

,Group,Customer_Segment,Sales_Before,Sales_After,Customer_Satisfaction_Before,Customer_Satisfaction_After,Purchase_Made
0,Control,High Value,240.548359,300.007568,74.684767,NaN,No
1,Treatment,High Value,246.862114,381.337555,100.000000,100.000000,Yes
2,Control,High Value,156.978084,179.330464,98.780735,100.000000,No
3,Control,Medium Value,192.126708,229.278031,49.333766,39.811841,Yes
4,NaN,High Value,229.685623,NaN,83.974852,87.738591,Yes


In [3]:
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    "Missing Values": missing_values,
    "Percentage (%)": missing_percentage
})

missing_df.sort_values(by="Missing Values", ascending=False)

,Missing Values,Percentage (%)
Customer_Segment,1966,19.66
Customer_Satisfaction_Before,1670,16.70
Customer_Satisfaction_After,1640,16.40
Sales_Before,1522,15.22
Group,1401,14.01
Purchase_Made,805,8.05
Sales_After,767,7.67


Identificér typer af datamangel og sætter tal på mangledne værdier

In [4]:
# Undersøg korrelation mellem missingness og andre variable
missing_indicator = df.isnull().astype(int)

correlation = missing_indicator.corr()
correlation

,Group,Customer_Segment,Sales_Before,Sales_After,Customer_Satisfaction_Before,Customer_Satisfaction_After,Purchase_Made
Group,1.000000,-0.016990,-0.008207,-0.012404,-0.008472,-0.012266,0.001291
Customer_Segment,-0.016990,1.000000,-0.000158,-0.003586,-0.017083,-0.000288,-0.011341
Sales_Before,-0.008207,-0.000158,1.000000,0.012828,0.011812,-0.003464,-0.008719
Sales_After,-0.012404,-0.003586,0.012828,1.000000,0.013008,-0.018052,-0.001027
Customer_Satisfaction_Before,-0.008472,-0.017083,0.011812,0.013008,1.000000,-0.010775,0.007455
Customer_Satisfaction_After,-0.012266,-0.000288,-0.003464,-0.018052,-0.010775,1.000000,-0.008954
Purchase_Made,0.001291,-0.011341,-0.008719,-0.001027,0.007455,-0.008954,1.000000


In [5]:
# Eksempel: antag sidste kolonne er target
target_column = df.columns[-1]

X = df.drop(columns=[target_column])
y = df[target_column]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
X_train_drop = X_train.dropna()
y_train_drop = y_train.loc[X_train_drop.index]

X_test_drop = X_test.dropna()
y_test_drop = y_test.loc[X_test_drop.index]

model_drop = RandomForestClassifier(random_state=42)
model_drop.fit(X_train_drop, y_train_drop)

y_pred_drop = model_drop.predict(X_test_drop)

print("Accuracy WITHOUT imputation:")
print(accuracy_score(y_test_drop, y_pred_drop))

ValueError: could not convert string to float: 'Treatment'

In [ ]:
mean_imputer = SimpleImputer(strategy="mean")

X_train_mean = mean_imputer.fit_transform(X_train)
X_test_mean = mean_imputer.transform(X_test)

model_mean = RandomForestClassifier(random_state=42)
model_mean.fit(X_train_mean, y_train)

y_pred_mean = model_mean.predict(X_test_mean)

print("Accuracy WITH Mean Imputation:")
print(accuracy_score(y_test, y_pred_mean))

In [ ]:
knn_imputer = KNNImputer(n_neighbors=5)

X_train_knn = knn_imputer.fit_transform(X_train)
X_test_knn = knn_imputer.transform(X_test)

model_knn = RandomForestClassifier(random_state=42)
model_knn.fit(X_train_knn, y_train)

y_pred_knn = model_knn.predict(X_test_knn)

print("Accuracy WITH KNN Imputation:")
print(accuracy_score(y_test, y_pred_knn))

In [ ]:
results = pd.DataFrame({
    "Method": ["Drop NaNs", "Mean Imputation", "KNN Imputation"],
    "Accuracy": [
        accuracy_score(y_test_drop, y_pred_drop),
        accuracy_score(y_test, y_pred_mean),
        accuracy_score(y_test, y_pred_knn)
    ]
})

results

In [ ]:
conn1 = sqlite3.connect("sales_database.db")

# Eksempel: gem salgsdata
df.to_sql("sales_table", conn1, if_exists="replace", index=False)

conn1.commit()
conn1.close()

In [ ]:
conn2 = sqlite3.connect("customer_database.db")

# Eksempel: opret kundetabel (tilpas kolonner)
customer_cols = [col for col in df.columns if "Customer" in col]

df_customers = df[customer_cols]

df_customers.to_sql("customer_table", conn2, if_exists="replace", index=False)

conn2.commit()
conn2.close()